# PhishShield — Detecção Visual de Interfaces Fraudulentas

**VGG16 + Transfer Learning + Grad-CAM + Gemini**  
Classifica screenshots de interfaces digitais como **legítimas** ou **fraudulentas** por análise exclusivamente visual.

| Acurácia (validação) | Recall — Phishing | Precisão | AUC-ROC |
|:---:|:---:|:---:|:---:|
| 70–72 % | **81,8 %** | 81,8 % | 0.75 |

---

### Estrutura do notebook

| # | Seção |
|---|---|
| 1 | Imports e configurações |
| 2 | Carregamento do dataset |
| 3 | Arquitetura do modelo (VGG16) |
| 4 | Treinamento |
| 5 | Avaliação (métricas, matriz de confusão, ROC) |
| 6 | Configuração da chave Gemini |
| 7 | Grad-CAM + análise em linguagem natural (Gemini) |

---

> **Ambiente:** Google Colab (GPU recomendada)  
> **Dataset esperado no Drive:** `MyDrive/dataset/fraudulenta/` e `MyDrive/dataset/legitima/`  
> **Modelo salvo em:** `MyDrive/modelo_tcc_vgg16_final.keras`


## 1 · Imports e Configurações

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import cv2
import seaborn as sns
from google.colab import drive
from tensorflow.keras.preprocessing import image
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# ── Hiperparâmetros globais ──────────────────────────────────────────────────
IMG_SIZE     = (224, 224)
BATCH_SIZE   = 32
EPOCHS       = 15
DATASET_PATH = '/content/drive/MyDrive/dataset/'
MODELO_PATH  = '/content/drive/MyDrive/modelo_tcc_vgg16_final.keras'


## 2 · Carregamento do Dataset

O dataset deve estar organizado no Google Drive no formato:

```
MyDrive/
└── dataset/
    ├── fraudulenta/   ← screenshots de páginas phishing
    └── legitima/      ← screenshots de páginas legítimas
```

Divisão automática: **80 % treino / 20 % validação**.  
Data Augmentation aplicado apenas ao conjunto de treino.


In [ ]:
drive.mount('/content/drive')

train_datagen = ImageDataGenerator(
    rescale          = 1./255,
    validation_split = 0.20,
    rotation_range   = 10,
    zoom_range       = 0.1,
    horizontal_flip  = False,
)

train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'training',
    shuffle     = True,
)

validation_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size = IMG_SIZE,
    batch_size  = BATCH_SIZE,
    class_mode  = 'binary',
    subset      = 'validation',
    shuffle     = False,
)

print(f"Classes detectadas: {train_generator.class_indices}")
print(f"Imagens de treino : {train_generator.samples}")
print(f"Imagens de validação: {validation_generator.samples}")


## 3 · Arquitetura do Modelo

**Backbone:** VGG16 pré-treinada no ImageNet.  
**Estratégia de Fine-Tuning:** Blocos 1–4 congelados; apenas o Bloco 5 é treinável.  
Isso preserva os filtros genéricos de visão e adapta apenas as camadas de alto nível semântico ao domínio de interfaces digitais.


In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Congela blocos 1–4; libera apenas o Bloco 5 (últimas 4 camadas conv)
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model = models.Sequential([
    base_model,
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])

model.compile(
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss      = 'binary_crossentropy',
    metrics   = [
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
    ],
)

model.summary()


## 4 · Treinamento

- **EarlyStopping** monitora `val_loss` com paciência de 5 épocas e restaura os melhores pesos automaticamente.  
- **ModelCheckpoint** salva o melhor estado do modelo no Google Drive.


In [ ]:
callbacks = [
    EarlyStopping(
        monitor            = 'val_loss',
        patience           = 5,
        restore_best_weights = True,
        verbose            = 1,
    ),
    ModelCheckpoint(
        filepath      = MODELO_PATH,
        monitor       = 'val_loss',
        save_best_only = True,
        verbose       = 1,
    ),
]

history = model.fit(
    train_generator,
    epochs          = EPOCHS,
    validation_data = validation_generator,
    callbacks       = callbacks,
)

melhor_epoca = np.argmin(history.history['val_loss']) + 1
print(f"\nMelhor época  : {melhor_epoca}")
print(f"Melhor val_loss: {min(history.history['val_loss']):.4f}")
print(f"Modelo salvo em: {MODELO_PATH}")


## 5 · Avaliação

Relatório de classificação, matriz de confusão, curvas de aprendizado e curva ROC.


In [ ]:
validation_generator.reset()
predictions  = model.predict(validation_generator)
y_pred       = (predictions >= 0.5).astype(int).flatten()
y_true       = validation_generator.classes
class_labels = list(validation_generator.class_indices.keys())

print("\n" + "=" * 88)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("=" * 88)
print(classification_report(y_true, y_pred, target_names=class_labels))

# ── Matriz de Confusão ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_labels, yticklabels=class_labels)
plt.title('Matriz de Confusão', fontsize=13, pad=15)
plt.ylabel('Verdadeiro', fontsize=11)
plt.xlabel('Predito', fontsize=11)
plt.tight_layout()
plt.show()

# ── Curvas de Aprendizado ────────────────────────────────────────────────────
melhor = np.argmin(history.history['val_loss'])
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'],     label='Treino',    color='steelblue',  linewidth=2)
plt.plot(history.history['val_loss'], label='Validação', color='darkorange', linewidth=2, linestyle='--')
plt.axvline(melhor, color='green', linestyle=':', label=f'Melhor época ({melhor + 1})')
plt.title('Evolução da Perda (Loss)', fontsize=12, pad=10)
plt.xlabel('Épocas'); plt.ylabel('Loss')
plt.legend(); plt.grid(True, linestyle=':', alpha=0.6)

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'],     label='Treino',    color='steelblue',  linewidth=2)
plt.plot(history.history['val_accuracy'], label='Validação', color='darkorange', linewidth=2, linestyle='--')
plt.axvline(melhor, color='green', linestyle=':', label=f'Melhor época ({melhor + 1})')
plt.title('Evolução da Acurácia', fontsize=12, pad=10)
plt.xlabel('Épocas'); plt.ylabel('Acurácia (%)')
plt.legend(); plt.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()

# ── Curva ROC ────────────────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(y_true, predictions.flatten())
roc_auc     = auc(fpr, tpr)

plt.figure(figsize=(6, 5.5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0]); plt.ylim([0.0, 1.05])
plt.xlabel('Taxa de Falsos Positivos (1 − Especificidade)', fontsize=11)
plt.ylabel('Taxa de Verdadeiros Positivos (Recall)', fontsize=11)
plt.title('Curva ROC', fontsize=13, pad=15)
plt.legend(loc='lower right', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()

# ── Distribuição absoluta de decisões ───────────────────────────────────────
tn, fp, fn, tp = cm.ravel()
categorias = [
    'Verdadeiro Negativo\n(Legítimo Correto)',
    'Falso Positivo\n(Alarme Falso)',
    'Falso Negativo\n(Phishing Não Detectado)',
    'Verdadeiro Positivo\n(Phishing Correto)',
]
cores = ['#2ca02c', '#ff7f0e', '#d62728', '#1f77b4']

plt.figure(figsize=(10, 5))
bars = plt.bar(categorias, [tn, fp, fn, tp], color=cores, edgecolor='black', alpha=0.85)
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 0.5,
             int(yval), ha='center', va='bottom', fontweight='bold')
plt.title('Distribuição Absoluta das Decisões do Modelo', fontsize=13, pad=15)
plt.ylabel('Quantidade de Imagens', fontsize=11)
plt.grid(axis='y', linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()


## 6 · Configuração da Chave Gemini

Necessária apenas para o módulo de explicabilidade (Seção 7).  
A chave é solicitada de forma segura via `getpass` e nunca fica exposta no notebook.


In [ ]:
import getpass

GEMINI_API_KEY = getpass.getpass("Cole sua chave da API Gemini: ")
print("Chave configurada.")


## 7 · Grad-CAM + Análise em Linguagem Natural (Gemini)

Para cada imagem analisada, o módulo:

1. Classifica a interface com o modelo VGG16 treinado
2. Gera um **heatmap Grad-CAM** destacando as regiões que influenciaram a decisão
3. Envia a imagem original + heatmap para o **Gemini 2.5 Flash**, que produz uma explicação em linguagem natural

Substitua `'caminho/para/screenshot.png'` pelo caminho real da imagem que deseja analisar.


In [ ]:
import base64, requests, tempfile
from tensorflow.keras.preprocessing import image as keras_image

def analisar_interface(img_path, model, api_key):
    """Classifica uma interface, gera Grad-CAM e obtém explicação do Gemini."""

    # ── Pré-processamento ────────────────────────────────────────────────────
    img_pil   = keras_image.load_img(img_path, target_size=(224, 224))
    img_array = keras_image.img_to_array(img_pil) / 255.0
    img_batch = tf.constant(np.expand_dims(img_array, axis=0), dtype=tf.float32)

    probabilidade = float(model(img_batch, training=False).numpy()[0][0])
    resultado  = "LEGÍTIMA"   if probabilidade >= 0.5 else "FRAUDULENTA"
    confianca  = probabilidade * 100 if probabilidade >= 0.5 else (1 - probabilidade) * 100
    cor_titulo = "green"      if probabilidade >= 0.5 else "red"

    # ── Grad-CAM ─────────────────────────────────────────────────────────────
    vgg            = model.layers[0]
    conv_layer     = vgg.get_layer('block5_conv3')
    vgg_conv_model = tf.keras.models.Model(
        inputs  = vgg.input,
        outputs = [conv_layer.output, vgg.output],
    )

    with tf.GradientTape() as tape:
        conv_out, vgg_out = vgg_conv_model(img_batch, training=False)
        tape.watch(conv_out)
        x = tf.reshape(vgg_out, [tf.shape(vgg_out)[0], -1])
        for layer in model.layers[1:]:
            x = layer(x, training=False)
        loss = x[:, 0]

    grads  = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0)
    if tf.reduce_max(heatmap) > 0:
        heatmap = heatmap / tf.reduce_max(heatmap)
    heatmap = heatmap.numpy()

    img_bgr    = cv2.resize(cv2.imread(img_path), (224, 224))
    hm_colored = cv2.applyColorMap(np.uint8(255 * cv2.resize(heatmap, (224, 224))), cv2.COLORMAP_JET)
    overlay    = cv2.cvtColor(cv2.addWeighted(img_bgr, 0.55, hm_colored, 0.45, 0), cv2.COLOR_BGR2RGB)

    # ── Visualização ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].imshow(img_pil);    axes[0].set_title('Interface analisada', fontsize=11);        axes[0].axis('off')
    axes[1].imshow(overlay);    axes[1].set_title('Grad-CAM (vermelho = alta influência)', fontsize=11); axes[1].axis('off')
    fig.suptitle(f"Classificação: {resultado}  |  Confiança: {confianca:.1f}%",
                 fontsize=13, fontweight='bold', color=cor_titulo)
    plt.tight_layout()

    tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
    plt.savefig(tmp.name, dpi=120, bbox_inches='tight')
    plt.show()
    tmp.close()

    # ── Análise Gemini ───────────────────────────────────────────────────────
    with open(tmp.name, 'rb') as f:
        img_b64 = base64.b64encode(f.read()).decode('utf-8')
    os.unlink(tmp.name)

    prompt = f"""Você é um especialista em segurança digital e visão computacional.

Estou te enviando duas imagens lado a lado:
- Esquerda: screenshot original da interface digital
- Direita: mapa Grad-CAM do modelo VGG16 (vermelho = alta atenção, azul = baixa atenção)

O modelo classificou esta interface como {resultado} com {confianca:.1f}% de confiança.

Com base nas imagens, explique em português de forma clara e acessível:
1. Quais regiões da interface o modelo focou mais e o que há nelas
2. O que esses elementos sugerem sobre a autenticidade da interface
3. Se a decisão do modelo parece coerente com o que você observa
4. Qualquer detalhe visual relevante que justifique ou questione a classificação"""

    response = requests.post(
        f"https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={api_key}",
        json={"contents": [{"parts": [
            {"text": prompt},
            {"inline_data": {"mime_type": "image/png", "data": img_b64}},
        ]}]},
    )

    if response.status_code == 200:
        analise = response.json()['candidates'][0]['content']['parts'][0]['text']
        print("\n" + "=" * 65)
        print("ANÁLISE DO GEMINI")
        print("=" * 65)
        print(analise)
        print("=" * 65)
    else:
        print(f"Erro na API Gemini ({response.status_code}): {response.text}")

    return resultado, confianca


# ── Uso ──────────────────────────────────────────────────────────────────────
# Warm-up para garantir que o grafo do modelo está construído
_ = model(np.zeros((1, 224, 224, 3), dtype=np.float32), training=False)

# Substitua pelo caminho real da imagem a ser analisada
analisar_interface('caminho/para/screenshot.png', model, GEMINI_API_KEY)
